# Feature Engineering Pipeline (Submission Version)
This notebook consolidates the final feature engineering workflow used in the thesis.

**Contents**
1. Setup & configuration
2. Baseline: original/raw feature importance (optional)
3. Strategy A: statistical + FFT features
4. Strategy B: psychologically-informed features
5. Controlled comparison (A vs B vs Combined)
6. Export outputs (CSVs + figures)

> Replace the folder paths in the config cell. The notebook is written to run with relative paths.


In [ ]:
# === 1) Setup & configuration ===
import os
import numpy as np
import pandas as pd

from dataclasses import dataclass
from typing import Dict, Optional, Tuple, List

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from sklearn.impute import SimpleImputer

from scipy.stats import skew, kurtosis, entropy
from scipy.fftpack import fft

SEED = 42
np.random.seed(SEED)

@dataclass
class Config:
    cheat_dir: str = "data/cheating"
    honest_dir: str = "data/non_cheating"
    out_dir: str = "outputs"
    test_size: float = 0.2

CFG = Config()

os.makedirs(CFG.out_dir, exist_ok=True)

print("Config:", CFG)


## 2) Data loading helpers
Each CSV file is treated as one session/trial. We create one row of engineered features per file.


In [ ]:
def list_csv_files(folder: str) -> List[str]:
    return [
        os.path.join(folder, f)
        for f in os.listdir(folder)
        if f.lower().endswith(".csv")
    ]

def load_csv(path: str) -> pd.DataFrame:
    # Robust CSV loading; adjust delimiter/encoding if needed.
    return pd.read_csv(path)

def load_labeled_filepaths(cheat_dir: str, honest_dir: str) -> pd.DataFrame:
    cheat = pd.DataFrame({"path": list_csv_files(cheat_dir), "label": 1})
    honest = pd.DataFrame({"path": list_csv_files(honest_dir), "label": 0})
    df = pd.concat([cheat, honest], ignore_index=True)
    if df.empty:
        raise ValueError("No CSV files found. Check CFG.cheat_dir / CFG.honest_dir.")
    return df

files_df = load_labeled_filepaths(CFG.cheat_dir, CFG.honest_dir)
files_df.head()


## 3) Strategy A: statistical + FFT features
Matches the thesis definitions: mean/std/min/max/skew/kurt + FFT energy/peak per numeric column.


In [ ]:
def to_numeric_df(df: pd.DataFrame, nan_col_thresh: float = 0.8) -> pd.DataFrame:
    # Convert commas to dots (German decimals), then coerce
    df = df.applymap(lambda x: str(x).replace(",", ".").strip() if isinstance(x, str) else x)
    num = df.apply(pd.to_numeric, errors="coerce")
    num = num.dropna(axis=1, thresh=int(len(num) * nan_col_thresh))
    num = num.ffill().bfill()
    return num

def extract_strategy_a(df: pd.DataFrame) -> Optional[Dict[str, float]]:
    num = to_numeric_df(df)
    if num.shape[1] == 0:
        return None
    feats: Dict[str, float] = {}
    for col in num.columns:
        s = num[col].values.astype(float)
        feats[f"{col}_mean"] = float(np.mean(s))
        feats[f"{col}_std"]  = float(np.std(s))
        feats[f"{col}_min"]  = float(np.min(s))
        feats[f"{col}_max"]  = float(np.max(s))
        feats[f"{col}_skew"] = float(skew(s))
        feats[f"{col}_kurt"] = float(kurtosis(s))
        try:
            fft_vals = np.abs(fft(s))[:len(s)//2]
            feats[f"{col}_fft_energy"] = float(np.sum(fft_vals**2))
            feats[f"{col}_fft_peak"]   = float(np.max(fft_vals))
        except Exception:
            feats[f"{col}_fft_energy"] = np.nan
            feats[f"{col}_fft_peak"]   = np.nan
    return feats

def build_feature_matrix(files: pd.DataFrame, extractor) -> pd.DataFrame:
    rows = []
    for _, r in files.iterrows():
        df = load_csv(r["path"])
        feats = extractor(df)
        if feats is None:
            continue
        feats["label"] = int(r["label"])
        feats["file"] = os.path.basename(r["path"])
        rows.append(feats)
    out = pd.DataFrame(rows)
    if out.empty:
        raise ValueError("No features extracted. Check input CSVs and extractor.")
    return out

A_df = build_feature_matrix(files_df, extract_strategy_a)
A_df.to_csv(os.path.join(CFG.out_dir, "features_strategyA.csv"), index=False)
A_df.shape


## 4) Strategy B: psychologically-informed features
This section is intentionally minimal and aligned with the thesis: average gaze, entropy, quadrant bias,
fixation/saccade counts and dispersion, scanpath length, dwell time, gaze efficiency, etc.

⚠️ You may need to adjust column names to match your CSV schema.


In [ ]:
# --- Column name mapping (edit to match your dataset) ---
COL_GAZE_X = "Gaze X"
COL_GAZE_Y = "Gaze Y"
COL_FIX_IDX = "Fixation Index"   # if available
COL_SAC_IDX = "Saccade Index"    # if available

def extract_strategy_b(df: pd.DataFrame, bins: int = 20) -> Optional[Dict[str, float]]:
    feats: Dict[str, float] = {}
    if COL_GAZE_X not in df.columns or COL_GAZE_Y not in df.columns:
        return None

    gaze_xy = df[[COL_GAZE_X, COL_GAZE_Y]].apply(pd.to_numeric, errors="coerce").dropna()
    if gaze_xy.empty:
        return None
    gx = gaze_xy[COL_GAZE_X].values
    gy = gaze_xy[COL_GAZE_Y].values

    feats["avg_gaze_x"] = float(np.mean(gx))
    feats["avg_gaze_y"] = float(np.mean(gy))

    # Gaze entropy via 2D histogram
    hist, _, _ = np.histogram2d(gx, gy, bins=bins)
    flat = hist.flatten()
    flat = flat[flat > 0]
    feats["gaze_entropy"] = float(entropy(flat)) if len(flat) else np.nan

    # Quadrant bias around medians
    mx, my = np.median(gx), np.median(gy)
    q_tl = np.mean((gx <= mx) & (gy >  my))
    q_tr = np.mean((gx >  mx) & (gy >  my))
    q_bl = np.mean((gx <= mx) & (gy <= my))
    q_br = np.mean((gx >  mx) & (gy <= my))
    feats["quadrant_bias_tl"] = float(q_tl)
    feats["quadrant_bias_tr"] = float(q_tr)
    feats["quadrant_bias_bl"] = float(q_bl)
    feats["quadrant_bias_br"] = float(q_br)

    # Fixation dispersion (std radius around centroid)
    cx, cy = np.mean(gx), np.mean(gy)
    feats["fixation_dispersion"] = float(np.sqrt(np.mean((gx - cx)**2 + (gy - cy)**2)))

    # Scanpath length (sum of step distances)
    dx = np.diff(gx); dy = np.diff(gy)
    step = np.sqrt(dx**2 + dy**2)
    feats["scanpath_length"] = float(np.sum(step))

    # Dwell time proxy (if timestamps exist, compute; else use sample count)
    feats["dwell_time_samples"] = float(len(gx))

    # Gaze transition rate proxy: sign changes relative to mean
    transitions = np.sum(np.diff((gx > np.mean(gx)).astype(int)) != 0)
    feats["gaze_transition_rate"] = float(transitions / max(len(gx), 1))

    # Optional: counts if event indices are available
    if COL_FIX_IDX in df.columns:
        fx = pd.to_numeric(df[COL_FIX_IDX], errors="coerce").dropna()
        feats["num_fixations"] = float(fx.nunique())
    if COL_SAC_IDX in df.columns:
        sx = pd.to_numeric(df[COL_SAC_IDX], errors="coerce").dropna()
        feats["num_saccades"] = float(sx.nunique())

    # Efficiency proxy: scanpath length / dwell
    feats["gaze_efficiency"] = float(feats["scanpath_length"] / max(feats["dwell_time_samples"], 1.0))

    return feats

B_df = build_feature_matrix(files_df, extract_strategy_b)
B_df.to_csv(os.path.join(CFG.out_dir, "features_strategyB.csv"), index=False)
B_df.shape


## 5) Controlled comparison (Strategy A vs Strategy B vs Combined)
We train the same Random Forest configuration and stratified split for all feature sets.


In [ ]:
def evaluate_feature_set(df: pd.DataFrame, label_col: str = "label", drop_cols: List[str] = ["file"]) -> Dict[str, float]:
    X = df.drop(columns=[label_col] + [c for c in drop_cols if c in df.columns], errors="ignore")
    y = df[label_col].astype(int).values

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=CFG.test_size, random_state=SEED, stratify=y
    )

    imputer = SimpleImputer(strategy="mean")
    X_train_imp = imputer.fit_transform(X_train)
    X_test_imp  = imputer.transform(X_test)

    clf = RandomForestClassifier(
        n_estimators=300,
        random_state=SEED,
        n_jobs=-1,
        class_weight="balanced"
    )
    clf.fit(X_train_imp, y_train)

    proba = clf.predict_proba(X_test_imp)[:, 1]
    pred = (proba >= 0.5).astype(int)

    metrics = {
        "accuracy": float(accuracy_score(y_test, pred)),
        "precision": float(precision_score(y_test, pred, zero_division=0)),
        "recall": float(recall_score(y_test, pred, zero_division=0)),
        "f1": float(f1_score(y_test, pred, zero_division=0)),
        "roc_auc": float(roc_auc_score(y_test, proba)),
    }
    return metrics

metrics_A = evaluate_feature_set(A_df)
metrics_B = evaluate_feature_set(B_df)

# Combined (inner join on file)
AB = pd.merge(
    A_df, B_df, on=["file", "label"], how="inner", suffixes=("_A", "_B")
)
metrics_AB = evaluate_feature_set(AB)

results = pd.DataFrame([
    {"model": "Strategy A (Statistical)", **metrics_A},
    {"model": "Strategy B (Psychological)", **metrics_B},
    {"model": "Combined", **metrics_AB},
])

results.to_csv(os.path.join(CFG.out_dir, "comparison_metrics.csv"), index=False)
results


## 6) Notes for submission
- Ensure `data/cheating` and `data/non_cheating` are included or provide instructions to obtain them.
- Keep outputs in `outputs/`.
- Record package versions (optional cell below).


In [ ]:
import sys, sklearn, scipy
print("Python:", sys.version)
print("numpy:", np.__version__)
print("pandas:", pd.__version__)
print("scikit-learn:", sklearn.__version__)
print("scipy:", scipy.__version__)
